# SOLVE_FIELD

This notebook can be used to determine the WCS solution of an OmegaCAM image.
It uses astrometry.net solve_field command.

Before executing solve_field the user must set certain variables:

- `export PATH=/opt/anaconda3/bin:$PATH`
- `export PYTHONHOME=/opt/anaconda3`
- `export PATH=/usr/local/astrometry/bin:$PATH`
- `export PATH=/usr/local/netpbm/bin:$PATH`

The output of this script is a directory with the name of the input FITS file, containing one subdirectory per extension with the output of solve-field, and with one FITS file with the same root name but with the string '\_wcs' appended between the root name and the '.fits' extension. This wcs file has nearly correct WCS keywords as determined by solve-field, and can then be input to sextractor and scamp via the fix_wcs python notebook.


In [ ]:
import os
import glob
import sys
import numpy as np
import subprocess
import time
import multiprocessing

# Set up matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

from astropy.io import fits

In [ ]:
def solvefield(file_name):
    
    dir_name = file_name[:-30]+'_solve_field'
    print('mef file:', dir_name[:-24]+'.fits')
    #print('file_name: ', file_name)
    #print('dir_name: ', dir_name)
    
    
    hdu_list = fits.open(file_name)        
    an_hdu = hdu_list[0]    
    crval1 = an_hdu.header['CRVAL1']
    crval2 = an_hdu.header['CRVAL2']
    crpix1 = an_hdu.header['CRPIX1']
    crpix2 = an_hdu.header['CRPIX2']
    hdu_list.close()
    
    # solve-field --ra 123.871525418909 --dec -49.1901672273134 --radius=1.0 --scale-low 0.02 --scale-high 0.25 ESO_CCD__66.fits
    pos_pars = '--ra '+ "%10.6f"%(crval1) + ' --dec ' + "%10.6f"%(crval2) + ' --radius=1.0 '
    scale_pars = ' --scale-low 0.02 --scale-high 0.25 --overwrite '
    wcs_pars = ' --crpix-x '+ "%10.6f"%(crpix1) + ' --crpix-y ' + "%10.6f"%(crpix2) + ' --no-tweak '
    solve_command_pars =  ' -D '+dir_name+ ' ' +pos_pars + scale_pars + wcs_pars + ' --new-fits ' + file_name[:-5]+'_newwcs.fits '+ file_name 
    solve_command_print =  ' -D ' + dir_name + \
                            '\n       ' + pos_pars + \
                            '\n     ' + scale_pars + \
                            '\n     ' + wcs_pars + \
                            '\n      --new-fits ' + file_name[:-5]+'_newwcs.fits '+ \
                            '\n      ' + file_name
    print(solve_command_print)
    #os.system('/usr/local/astrometry/bin/solve-field '+solve_command_pars)
    logfile = file_name[:-5]+'_subprocess.log'
    output = subprocess.check_output([
        '/usr/local/astrometry/bin/solve-field', 
        '-D', dir_name,
        '--ra', "%10.6f"%(crval1),
        '--dec', "%10.6f"%(crval2),
        '--radius', '1.0',  
        '--scale-low', '0.02',
        '--scale-high', '0.25',
        '--overwrite',
        '--crpix-x', "%10.6f"%(crpix1),
        '--crpix-y', "%10.6f"%(crpix2),
        '--no-tweak',
        '--new-fits', file_name[:-5]+'_newwcs.fits ',
        file_name])
    
    print('appending output to: ', logfile)
    output_str = output.decode()
    for line in output_str.split('\n'):
        with open(logfile, "a") as f:
            print(line, file=f)
    f.close()
    #print('solve-field '+ext_file_name)
    

In [ ]:
def run_solve_field(argv):

    filename = argv+'.fits'
    dir_name = os.path.abspath(argv)
    try:
        # Create target Directory
        os.mkdir(dir_name)
        print("Directory " , dir_name ,  " Created ") 
    except FileExistsError:
        print("Directory " , dir_name ,  " already exists")

    # here we create a directory where we will save all data from solve-field
    hdu_list = fits.open(filename)
    list_of_extensions = []
    list_of_filenames = []
    for an_hdu in  hdu_list[1:]:
        list_of_extensions.append(an_hdu.header['EXTNAME'])
        ext_file_name = an_hdu.header['EXTNAME'].replace('#','_')
        #print(ext_file_name, an_hdu.header['CRVAL1'], an_hdu.header['CRVAL2'])
        solve_field_dir_name = dir_name+'/'+ext_file_name+'_solve_field/'
        try:
            # Create target Directory
            os.mkdir(solve_field_dir_name)
            print("solve-field directory " , solve_field_dir_name ,  " Created ") 
        except FileExistsError:
            print("solve-field directory " , solve_field_dir_name ,  " already exists")
        list_of_filenames.append(solve_field_dir_name+'/'+ext_file_name+'.fits')
        fits.writeto(solve_field_dir_name+'/'+ext_file_name+'.fits', an_hdu.data, an_hdu.header, overwrite=True)
    hdu_list.close()
    
    
    
    #list_of_extensions = ["ESO_CCD_#%2d" % (i,) for i in range(65, 97)]
    processlist = list()
    for xtension in list_of_extensions:
        file_name = solve_field_dir_name+'/'+ext_file_name+'.fits'
        print(filename)
        print('Adding extension: %s to processpool: ' % (xtension,))
        processlist.append([file_name])
        print("added...")
    print('number of extensions: ', len(list_of_filenames))
    cpu_count = 32
    pool = multiprocessing.Pool(processes = cpu_count)
    num_tasks = len(processlist)
    print('number of tasks: ', num_tasks)
    processresult = pool.imap_unordered(solvefield,list_of_filenames)
    pool.close()
    
    print("run_solvefield: %i tasks"%num_tasks)
    while (True):
        time.sleep(5)
        completed = processresult._index
        if completed == num_tasks:
            output = ""
            sys.stdout.write("\r\x1b[K"+output.__str__())
            break
        output = "\r Waiting for %i tasks to complete (%i%% done) ..." %(num_tasks-completed, float(completed)/float(num_tasks)*100.0)
        sys.stdout.write("\r\x1b[K"+output.__str__())
        sys.stdout.flush()
    print("done.")

In [ ]:
#file_name = 'OMEGA.2014-03-10T01:39:59.047_pre.fits'
reduced_dir = '/home/fselman/omegacam/fernando/reduced/science/CP_Pup_081146p060_m352105p0/NB_659/2017-03-13T01:57:49/'
current_dir = '/home/fselman/omegacam/fernando/'
print('current directory:', os.getcwd())
os.chdir(reduced_dir)
print('current directory:', os.getcwd())
list_to_solve = glob.glob('OMEGA.*_pre.fits')
if len(list_to_solve) > 0:
    i = 0
    for file_name in list_to_solve:
        i += 1
        print('solving %i/%i: %s'%(i, len(list_to_solve), file_name[:-5]+'.fits',))
        run_solve_field(file_name[:-5])
print('current directory:', os.getcwd())
os.chdir(current_dir)
print('current directory:', os.getcwd())


